============================================================
PYSPARK FUNDAMENTALS - Interview Q&A with Code
============================================================
Swiss Re Interview Prep: Senior Application Engineer
Topic: Core Spark Concepts
============================================================

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum, avg, count, when, lit
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType


# Q1: What is the difference between RDD, DataFrame, and Dataset?


ANSWER:

| Feature        | RDD                    | DataFrame              | Dataset               |
|----------------|------------------------|------------------------|-----------------------|
| Type Safety    | Yes (compile-time)     | No (runtime only)      | Yes (compile-time)    |
| Optimization   | No Catalyst/Tungsten   | Yes, fully optimized   | Yes, fully optimized  |
| Schema         | No schema              | Has schema             | Has schema            |
| API            | Functional (map, etc.) | SQL-like               | Mix of both           |
| Python Support | Yes                    | Yes                    | No (Scala/Java only)  |

WHEN TO USE:
- RDD: Low-level transformations, unstructured data, legacy code
- DataFrame: 99% of PySpark work, SQL queries, structured data (USE THIS!)
- Dataset: Scala/Java projects needing type safety

INTERVIEW TIP: "In Python, I always use DataFrames because Datasets are not available,
and DataFrames provide better optimization through Catalyst and Tungsten engines."


In [ ]:
def rdd_vs_dataframe_demo():
    spark = SparkSession.builder.appName("Fundamentals").master("local[*]").getOrCreate()
    
    # RDD Approach (Old, less optimized)
    rdd = spark.sparkContext.parallelize([(1, "Alice", 100), (2, "Bob", 200)])
    rdd_sum = rdd.map(lambda x: x[2]).reduce(lambda a, b: a + b)
    print(f"RDD Sum: {rdd_sum}")
    
    # DataFrame Approach (Modern, optimized)
    df = spark.createDataFrame([(1, "Alice", 100), (2, "Bob", 200)], ["id", "name", "amount"])
    df_sum = df.agg(sum("amount")).collect()[0][0]
    print(f"DataFrame Sum: {df_sum}")
    
    spark.stop()


In [ ]:
rdd_vs_dataframe_demo()

# Q2: Explain Lazy Evaluation in Spark


ANSWER:

Spark uses LAZY EVALUATION, meaning:
- Transformations (filter, map, join) are NOT executed immediately
- They build a Directed Acyclic Graph (DAG) of operations
- Execution only happens when an ACTION is called (collect, count, show, write)

WHY IS THIS GOOD?
1. Optimization: Spark can optimize the entire plan before running
2. Efficiency: Avoids unnecessary computation
3. Fault Tolerance: DAG can be re-executed from checkpoints

TRANSFORMATIONS (Lazy):
- select, filter, join, groupBy, withColumn, drop, distinct

ACTIONS (Trigger Execution):
- show, collect, count, write, take, first, reduce

INTERVIEW TIP: "I always chain my transformations and call an action at the end.
This lets Spark optimize the entire pipeline."


In [ ]:
def lazy_evaluation_demo():
    spark = SparkSession.builder.appName("LazyDemo").master("local[*]").getOrCreate()
    
    df = spark.range(0, 1000000)
    
    # These do NOT execute yet (Lazy)
    filtered = df.filter(col("id") > 500000)
    doubled = filtered.withColumn("doubled", col("id") * 2)
    
    print("No execution yet... transformations are lazy.")
    
    # THIS triggers execution (Action)
    result = doubled.count()
    print(f"Count (Action triggered): {result}")
    
    spark.stop()


In [ ]:
lazy_evaluation_demo()

# Q3: What are Narrow vs Wide Transformations?


ANSWER:

NARROW TRANSFORMATIONS:
- Data stays within the same partition
- No data movement across the network (no shuffle)
- Examples: map, filter, select, withColumn

WIDE TRANSFORMATIONS:
- Data needs to move across partitions (SHUFFLE)
- Expensive operation (network I/O)
- Examples: groupBy, join, distinct, repartition, orderBy

WHY DOES THIS MATTER?
- Shuffles are the #1 cause of slow Spark jobs
- Shuffles write data to disk and transfer over network
- GOAL: Minimize wide transformations

INTERVIEW TIP: "I always try to filter data as early as possible (narrow)
before doing any joins or aggregations (wide) to reduce shuffle size."


In [ ]:
def narrow_vs_wide_demo():
    spark = SparkSession.builder.appName("ShuffleDemo").master("local[*]").getOrCreate()
    
    df = spark.range(0, 10000).withColumn("key", col("id") % 10)
    
    # Narrow (No Shuffle) - Fast
    narrow_result = df.filter(col("id") > 5000).withColumn("new_col", col("id") * 2)
    
    # Wide (Shuffle Required) - Slow
    wide_result = df.groupBy("key").agg(sum("id").alias("total"))
    
    # Check execution plan
    print("--- Narrow Transformation Plan ---")
    narrow_result.explain()
    
    print("\n--- Wide Transformation Plan (Look for 'Exchange' = Shuffle) ---")
    wide_result.explain()
    
    spark.stop()


In [ ]:
narrow_vs_wide_demo()

# Q4: How does Spark handle data partitioning?


ANSWER:

PARTITIONING is how Spark divides data across executors.

KEY CONCEPTS:
1. spark.sql.shuffle.partitions: Default 200 (controls shuffle output partitions)
2. repartition(n): Forces a full shuffle to create n partitions
3. coalesce(n): Reduces partitions WITHOUT a full shuffle (merge only)

BEST PRACTICES:
- Too few partitions: Executor OOM, slow processing
- Too many partitions: Overhead, many small files
- Rule of Thumb: 2-4 partitions per CPU core available

INTERVIEW TIP: "I tune shuffle partitions based on the data size.
For small data (< 100MB), I use coalesce(1). For large data, I set
spark.sql.shuffle.partitions to match the parallelism of the cluster."


In [ ]:
def partitioning_demo():
    spark = SparkSession.builder.appName("PartitionDemo").master("local[*]").getOrCreate()
    
    df = spark.range(0, 10000)
    
    print(f"Initial Partitions: {df.rdd.getNumPartitions()}")
    
    # Repartition (Forces Shuffle)
    repartitioned = df.repartition(8)
    print(f"After repartition(8): {repartitioned.rdd.getNumPartitions()}")
    
    # Coalesce (No Shuffle, just merges)
    coalesced = repartitioned.coalesce(2)
    print(f"After coalesce(2): {coalesced.rdd.getNumPartitions()}")
    
    spark.stop()


In [ ]:
partitioning_demo()

# Q5: What is the Catalyst Optimizer?


ANSWER:

Catalyst is Spark SQL's query optimizer. It:
1. Parses SQL/DataFrame code into an Abstract Syntax Tree (AST)
2. Analyzes the AST and resolves column names/types
3. Applies Logical Optimization (predicate pushdown, constant folding)
4. Applies Physical Optimization (chooses join strategies)
5. Generates optimized Java bytecode (via Tungsten)

KEY OPTIMIZATIONS:
- Predicate Pushdown: Filters pushed to data source (e.g., Parquet)
- Projection Pruning: Only reads required columns
- Join Reordering: Puts smaller tables first in joins

INTERVIEW TIP: "Catalyst is why DataFrames are faster than RDDs.
I can see the optimized plan using df.explain(True)."


In [ ]:
def catalyst_demo():
    spark = SparkSession.builder.appName("CatalystDemo").master("local[*]").getOrCreate()
    
    df = spark.range(0, 1000).toDF("id")
    
    # Complex query
    result = df.filter(col("id") > 100).filter(col("id") < 500).select("id")
    
    # See how Catalyst optimizes this
    print("--- Extended Plan (Shows Catalyst Optimization) ---")
    result.explain(True)
    
    spark.stop()


In [ ]:
catalyst_demo()

In [ ]:
if __name__ == "__main__":
    print("=" * 60)
    print("Q1: RDD vs DataFrame Demo")
    print("=" * 60)
    rdd_vs_dataframe_demo()
    
    print("\n" + "=" * 60)
    print("Q2: Lazy Evaluation Demo")
    print("=" * 60)
    lazy_evaluation_demo()
    
    print("\n" + "=" * 60)
    print("Q3: Narrow vs Wide Transformations Demo")
    print("=" * 60)
    narrow_vs_wide_demo()
    
    print("\n" + "=" * 60)
    print("Q4: Partitioning Demo")
    print("=" * 60)
    partitioning_demo()
    
    print("\n" + "=" * 60)
    print("Q5: Catalyst Optimizer Demo")
    print("=" * 60)
    catalyst_demo()
